# Preprocess Images
### Purpose: TIFF → PNG and refresh annotation paths.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

from src.data.image_loader import validate_image_directory
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t
from src.utils.image_converter import ImageConverter

config = Config.load(root=root)
init_notebook(config.train.seed)



=== init_notebook ===
Done


#### Validate → convert → update annotations → re-validate


In [2]:
t("Validating train images")
train_results = validate_image_directory(config.paths.train_images)

t("Converting train images")
converter = ImageConverter(
    source_dir=config.paths.train_images,
    target_dir=config.paths.train_images,
)
stats = converter.convert_batch(overwrite=False)

if config.paths.eval_images and config.paths.eval_images.exists():
    t("Converting eval images")
    eval_stats = ImageConverter(
        source_dir=config.paths.eval_images,
        target_dir=config.paths.eval_images,
    ).convert_batch(overwrite=False)

if config.paths.annotations and Path(config.paths.annotations).exists():
    t("Updating annotations to reference PNG files")
    converter.update_annotations(
        annotations_path=config.paths.annotations,
        create_backup=True,
    )

t("Validating converted images")
final_results = validate_image_directory(config.paths.train_images)
t("Conversion Summary")
p("train validate", train_results)
p("train convert", stats)
p("final validate", final_results)
if config.paths.eval_images and config.paths.eval_images.exists():
    p("eval validate", validate_image_directory(config.paths.eval_images))


=== Validating train images ===
Validating images: 0 files
Valid images: 0
Invalid images: 0
=== Converting train images ===
[Warn]: No TIFF files found in source directory
=== Converting eval images ===
[Warn]: No TIFF files found in source directory
=== Updating annotations to reference PNG files ===
[Info]: Created backup: train_annotations.json.backup_20260924_123501
[Info]: Latest backup: train_annotations.json.backup
[Info]: Updated 0 file references in annotations
=== Validating converted images ===
Validating images: 0 files
Valid images: 0
Invalid images: 0
=== Conversion Summary ===
train validate: 4 keys
  total: 0
  valid: 0
  invalid: 0
  problematic_files: []
train convert: 4 keys
  total: 0
  converted: 0
  skipped: 0
  failed: 0
final validate: 4 keys
  total: 0
  valid: 0
  invalid: 0
  problematic_files: []
Validating images: 0 files
Valid images: 0
Invalid images: 0
eval validate: 4 keys
  total: 0
  valid: 0
  invalid: 0
  problematic_files: []
